In [1]:
from fedn.utils.helpers.helpers import get_helper
from ultralytics import YOLO
import torch
import collections
import numpy as np
import hopsworks
import os
from PIL import Image
import train

HELPER_MODULE = "numpyhelper"
helper = get_helper(HELPER_MODULE)

In [2]:
train.copy_to_local_dir_training_data()
model = train.load_parameters("weights/face_finder_best.npz")
params = {
    'data': '/hopsfs/Jupyter/yolov8-face/data/widerface.yaml',
    'epochs': 1,
    'batch': 32,
    'imgsz': 640,
    'device': 0,
    'resume': False,
    'workers': 0,
    'cache': "ram",
    'amp': True,
}

print(params)

Copying training data to local directory.
Finished copying.
WARNING ⚠️ no model scale passed. Assuming scale='n'.
{'data': '/hopsfs/Jupyter/yolov8-face/data/widerface.yaml', 'epochs': 1, 'batch': 32, 'imgsz': 640, 'device': 0, 'resume': False, 'workers': 0, 'cache': 'ram', 'amp': True}


In [3]:
model.train(**params)

New https://pypi.org/project/ultralytics/8.3.192 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.149 🚀 Python-3.10.11 torch-2.5.1+cu121 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=ram, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/hopsfs/Jupyter/yolov8-face/data/widerface.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=1, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=model.yaml, momentum=0.937, mosaic=1.0, multi_scale=False, name=train7, nbs=64, nms=False, opset=None,

train: Scanning /tmp/widerface/train... 2464 images, 0 backgrounds, 0 corrupt: 100%|██████████| 2464/2464 [00:01<00:00, 1329.21it/s]


train: New cache created: /tmp/widerface/train.cache
WARNING ⚠️ cache='ram' may produce non-deterministic training results. Consider cache='disk' as a deterministic alternative if your disk space allows.


train: Caching images (2.0GB RAM): 100%|██████████| 2464/2464 [00:17<00:00, 143.26it/s]

val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1882.4±537.4 MB/s, size: 90.7 KB)


...

val: Caching images (0.5GB RAM): 100%|██████████| 617/617 [00:04<00:00, 141.17it/s]


2025-09-03 18:04:19,313 WARNING: DeprecationWarning: backend2gui is deprecated since IPython 8.24, backends are managed in matplotlib and can be externally registered.

Plotting labels to runs/detect/train7/labels.jpg... 
2025-09-03 18:04:19,342 WARNING: DeprecationWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead

2025-09-03 18:04:19,343 WARNING: DeprecationWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead

2025-09-03 18:04:19,345 WARNING: DeprecationWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead

2025-09-03 18:04:19,346 WARNING: DeprecationWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead

2025-09-03 18:04:19,347 WARNING: DeprecationWarning: b

        1/1      13.4G      1.417     0.7262      1.032        570        640: 100%|██████████| 77/77 [00:59<00:00,  1.30it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95):  40%|████      | 4/10 [00:06<00:10,  1.73s/it]Exception in thread Thread-37 (plot_images):
Traceback (most recent call last):
  File "/srv/hops/anaconda/envs/hopsworks_environment/lib/python3.10/threading.py", line 1016, in _bootstrap_inner
    self.run()
  File "/srv/hops/anaconda/envs/hopsworks_environment/lib/python3.10/site-packages/ipykernel/ipkernel.py", line 766, in run_closure
    _threading_Thread_run(self)
  File "/srv/hops/anaconda/envs/hopsworks_environment/lib/python3.10/threading.py", line 953, in run
    self._target(*self._args, **self._kwargs)
  File "/srv/hops/anaconda/envs/hopsworks_environment/lib/python3.10/site-packages/ultralytics/utils/plotting.py", line 845, in plot_images
    annotator.im.save(fname)  # save
  File "/srv/hops/anaconda/envs/hopswor

                   all        617       7891      0.854      0.571      0.656      0.365


OSError: [Errno 5] Input/output error

### Model Evaluation
Predict bounding boxes for an example image and save the output image with the model.

In [ ]:
mr = hopsworks.login().get_model_registry()

model_dir = "mr_model"
os.makedirs(f"{model_dir}/images", exist_ok=True)
train.save_parameters(model, f"./{model_dir}/fine-tuned-model.npz")


img_path = "data/images/bus.jpg"
results = model.predict(
    img_path,
    imgsz=640,
    conf=0.75,
    iou=0.7,
    device=0,
    verbose=False
)

img = results[0].plot()  # BGR numpy array
img = Image.fromarray(img[..., ::-1])  # Convert to RGB for PIL

base, _ = os.path.splitext(os.path.basename(img_path))
output_filename = f"./{model_dir}/images/{base}-faces-detected.png"
output_path = os.path.abspath(output_filename)
img.save(output_path, format="PNG")

### Save Trained Model to Hopsworks Model Registry

Save the serialized model, evaluation images, and any metrics to the model registry

In [ ]:
metrics = {
    "epochs": params['epochs'],
    "batch": params['batch'],    
}

faces_model = mr.python.create_model(
    name="facerecognition", 
    metrics=metrics,
    description="Yolo-v8 face recognition model", 
)

# Save the model to the specified directory
faces_model.save(model_dir)